In [ ]:
# Cellule 1 : Paramètres de connexion
server = 'localhost'  # ou 'DESKTOP-3U42S6P'
database = 'event_DWH'

In [ ]:
# Cellule 2 : Essayer de se connecter sans identifiants (Windows Authentication)
from sqlalchemy import create_engine

# Version Windows Authentication (utilise ton utilisateur Windows)
connection_string = f"mssql+pyodbc://@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"

print("Tentative de connexion...")
print(connection_string)

try:
    engine = create_engine(connection_string)
    connection = engine.connect()
    print("✅ CONNEXION RÉUSSIE !")
    connection.close()
except Exception as e:
    print(f"❌ Erreur: {e}")

In [ ]:
# Cellule 3 : Tester la lecture d'une table
import pandas as pd

# Test sur une petite table
df_test = pd.read_sql("SELECT TOP 5 * FROM Dim_Beneficiary", engine)

print("✅ Données chargées !")
print(f"Shape: {df_test.shape}")
print("\nAperçu:")
print(df_test.head())

In [ ]:
# Cellule 4 : Compter les lignes
count_query = "SELECT COUNT(*) as total FROM FACT_VENTES"
total = pd.read_sql(count_query, engine)

print(f"📊 Nombre total de réservations: {total['total'][0]:,} lignes")

In [ ]:
# Cellule : Chargement complet CORRIGÉ
print("⏳ Chargement des données...")

query = """
SELECT 
    f.id_reservation,
    f.price,
    f.nbr_reservations,
    f.nbr_visitors,
    f.marketing_spend,
    f.market_count,
    f.status as reservation_status,
    e.title as event_title,
    e.type as event_type,
    e.event_date,
    cat.name as category_name,
    l.city,
    l.country,
    ev.rating,
    ev.comment,
    t.trend_score,
    t.growth_rate_pct,
    v.capacity_min,
    v.capacity_max,
    v.venue_type,
    ent.name as entertainer_name,
    ent.categorie as entertainer_category,
    ent.nationality,
    ent.followers_instagram,
    ent.followers_tiktok,
    ent.average_price as entertainer_price
FROM FACT_VENTES f
LEFT JOIN Dim_Event e ON f.id_event = e.id_event
LEFT JOIN Dim_Category cat ON f.id_category = cat.id_category
LEFT JOIN Dim_Localisation l ON f.id_localisation = l.id_localisation
LEFT JOIN Dim_Evaluation ev ON f.id_evaluation = ev.id_evaluation
LEFT JOIN Dim_Trends t ON f.id_trend = t.id_trend
LEFT JOIN Dim_Venue v ON f.id_venue = v.id_venue
LEFT JOIN Dim_Entertainer ent ON f.id_entertainer = ent.id_entertainer
"""

df = pd.read_sql(query, engine)

print(f"✅ Chargé: {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"\n📋 Colonnes disponibles:")
print(df.columns.tolist())


# Sauvegarde en CSV
df.to_csv('event_data.csv', index=False)
print("✅ Données sauvegardées dans 'event_data.csv'")

# Aperçu
print("\n📊 Aperçu:")
print(df.head())

In [ ]:
# Cellule : Aperçu des données
print("📊 Aperçu:")
print(df.head())

print("\n🔢 Types de données:")
print(df.dtypes)

print("\n📊 Stats rapides:")
print(df.describe())

In [ ]:
# Cellule : Info générale
print(f"📊 Taille: {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"\n📋 Liste des colonnes:")
for i, col in enumerate(df.columns):
    print(f"   {i+1}. {col}")

In [ ]:
# Cellule : Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Manquantes': missing, 'Pourcentage': missing_pct})
missing_df = missing_df[missing_df['Manquantes'] > 0].sort_values('Pourcentage', ascending=False)

print("🔍 Valeurs manquantes:")
if len(missing_df) > 0:
    print(missing_df)
else:
    print("✅ Aucune valeur manquante !")

In [ ]:
# Cellule : Identifier et retirer les colonnes ID
# Colonnes à exclure de l'analyse statistique
id_columns = ['id_reservation', 'id_event', 'id_category', 'id_localisation', 
              'id_evaluation', 'id_trend', 'id_venue', 'id_entertainer']

# Vérifier lesquelles existent vraiment
id_columns_exist = [col for col in id_columns if col in df.columns]

print(f"🔍 Colonnes ID à exclure: {id_columns_exist}")

# Créer un DataFrame sans les ID pour l'analyse
df_clean = df.drop(columns=id_columns_exist)

print(f"\n✅ DataFrame nettoyé: {df_clean.shape[0]} lignes, {df_clean.shape[1]} colonnes")
print(f"   (suppression de {len(id_columns_exist)} colonnes ID)")

In [ ]:
# Cellule : Statistiques sur les colonnes numériques (sans les ID)
print("📊 Statistiques des colonnes numériques (sans ID):")
numeric_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns
print(f"Colonnes numériques analysées: {numeric_cols.tolist()}")
print("\n")
df_clean[numeric_cols].describe()

In [ ]:
# Cellule : Voir les colonnes restantes
print("📋 Toutes les colonnes après nettoyage:")
for i, col in enumerate(df_clean.columns):
    print(f"   {i+1}. {col} ({df_clean[col].dtype})")

In [ ]:
# Cellule : Valeurs manquantes sur le DataFrame nettoyé
missing = df_clean.isnull().sum()
missing_pct = (missing / len(df_clean)) * 100
missing_df = pd.DataFrame({'Manquantes': missing, 'Pourcentage': missing_pct})
missing_df = missing_df[missing_df['Manquantes'] > 0].sort_values('Pourcentage', ascending=False)

print("🔍 Valeurs manquantes après nettoyage:")
if len(missing_df) > 0:
    print(missing_df)
else:
    print("✅ Aucune valeur manquante !")

In [ ]:
# Cellule : Distribution des réservations
print("📈 Distribution du nombre de réservations:")
print(df['nbr_reservations'].describe())

# Créer la target (client fidèle = plus de 2 réservations)
df['target_fidele'] = (df['nbr_reservations'] > 2).astype(int)
print(f"\n🎯 Clients fidèles (target=1): {df['target_fidele'].sum():.0f}")
print(f"   Clients non fidèles (target=0): {(df['target_fidele'] == 0).sum():.0f}")
print(f"   Taux de fidélité: {df['target_fidele'].mean()*100:.1f}%")

In [ ]:
# ============================================
# CLASSIFICATION AVEC PIPELINE + GRIDSEARCH (CORRIGÉE)
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, f1_score, accuracy_score

# Modèles
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

print("="*60)
print("📊 CLASSIFICATION - AVEC PIPELINE + GRIDSEARCH")
print("="*60)

# ============================================
# 1. CHARGEMENT DES DONNÉES
# ============================================
df = pd.read_csv('event_data.csv')

# 🔧 CORRECTION: Target basée sur la médiane (pas sur >=3)
median_reservations = df['nbr_reservations'].median()
df['target'] = (df['nbr_reservations'] > median_reservations).astype(int)

print(f"📊 Médiane des réservations: {median_reservations}")
print(f"🎯 Distribution target:")
print(f"   Classe 1 (fidèle): {df['target'].sum()} ({df['target'].mean()*100:.1f}%)")
print(f"   Classe 0 (non fidèle): {(df['target']==0).sum()} ({(1-df['target'].mean())*100:.1f}%)")

# Vérifier qu'on a bien 2 classes
if df['target'].nunique() < 2:
    raise ValueError("❌ Impossible de faire de la classification: une seule classe détectée!")

# Features
feature_cols = ['price', 'nbr_visitors', 'marketing_spend', 'market_count', 
                'trend_score', 'rating', 'followers_instagram', 'average_price']
feature_cols = [col for col in feature_cols if col in df.columns]

X = df[feature_cols].copy()
y = df['target'].copy()

# Gestion des NaN
for col in X.columns:
    if X[col].isnull().sum() > 0:
        X[col].fillna(X[col].median(), inplace=True)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Train: {X_train.shape}, Test: {X_test.shape}")
print(f"🎯 Target train: {y_train.mean()*100:.1f}% de fidèles")

# ============================================
# 2. PIPELINE + GRIDSEARCH POUR CHAQUE MODÈLE
# ============================================

results = {}

# -------------------------------------------------
# MODÈLE 1: Decision Tree
# -------------------------------------------------
print("\n" + "="*50)
print("🌳 1. DECISION TREE")
print("="*50)

pipeline_dt = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

param_grid_dt = {
    'classifier__max_depth': [3, 5, 10],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2]
}

grid_dt = GridSearchCV(pipeline_dt, param_grid_dt, cv=5, scoring='f1', n_jobs=-1)
grid_dt.fit(X_train, y_train)

print(f"✅ Meilleurs paramètres: {grid_dt.best_params_}")
print(f"✅ Meilleur F1 (CV): {grid_dt.best_score_:.4f}")

y_pred_dt = grid_dt.predict(X_test)

# Vérifier que predict_proba fonctionne (2 classes)
if len(grid_dt.classes_) == 2:
    y_proba_dt = grid_dt.predict_proba(X_test)[:, 1]
    auc_dt = roc_auc_score(y_test, y_proba_dt)
    print(f"📊 ROC-AUC: {auc_dt:.4f}")
else:
    auc_dt = 0.5
    print("⚠️ Une seule classe détectée, ROC-AUC non applicable")

f1_dt = f1_score(y_test, y_pred_dt)
print(f"📊 Test - Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}, F1: {f1_dt:.4f}")

results['Decision Tree'] = {'f1': f1_dt, 'auc': auc_dt, 'model': grid_dt, 'y_pred': y_pred_dt}

# -------------------------------------------------
# MODÈLE 2: Random Forest
# -------------------------------------------------
print("\n" + "="*50)
print("🌲 2. RANDOM FOREST")
print("="*50)

pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10],
    'classifier__min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(pipeline_rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1)
grid_rf.fit(X_train, y_train)

print(f"✅ Meilleurs paramètres: {grid_rf.best_params_}")
print(f"✅ Meilleur F1 (CV): {grid_rf.best_score_:.4f}")

y_pred_rf = grid_rf.predict(X_test)

if len(grid_rf.classes_) == 2:
    y_proba_rf = grid_rf.predict_proba(X_test)[:, 1]
    auc_rf = roc_auc_score(y_test, y_proba_rf)
    print(f"📊 ROC-AUC: {auc_rf:.4f}")
else:
    auc_rf = 0.5

f1_rf = f1_score(y_test, y_pred_rf)
print(f"📊 Test - Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}, F1: {f1_rf:.4f}")

results['Random Forest'] = {'f1': f1_rf, 'auc': auc_rf, 'model': grid_rf, 'y_pred': y_pred_rf}

# -------------------------------------------------
# MODÈLE 3: XGBoost
# -------------------------------------------------
print("\n" + "="*50)
print("🚀 3. XGBOOST")
print("="*50)

pipeline_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
])

param_grid_xgb = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [3, 5],
    'classifier__learning_rate': [0.01, 0.1]
}

grid_xgb = GridSearchCV(pipeline_xgb, param_grid_xgb, cv=5, scoring='f1', n_jobs=-1)
grid_xgb.fit(X_train, y_train)

print(f"✅ Meilleurs paramètres: {grid_xgb.best_params_}")
print(f"✅ Meilleur F1 (CV): {grid_xgb.best_score_:.4f}")

y_pred_xgb = grid_xgb.predict(X_test)

if len(grid_xgb.classes_) == 2:
    y_proba_xgb = grid_xgb.predict_proba(X_test)[:, 1]
    auc_xgb = roc_auc_score(y_test, y_proba_xgb)
    print(f"📊 ROC-AUC: {auc_xgb:.4f}")
else:
    auc_xgb = 0.5

f1_xgb = f1_score(y_test, y_pred_xgb)
print(f"📊 Test - Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}, F1: {f1_xgb:.4f}")

results['XGBoost'] = {'f1': f1_xgb, 'auc': auc_xgb, 'model': grid_xgb, 'y_pred': y_pred_xgb}

# -------------------------------------------------
# MODÈLE 4: SVM
# -------------------------------------------------
print("\n" + "="*50)
print("⚡ 4. SVM")
print("="*50)

pipeline_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(random_state=42, probability=True))
])

param_grid_svm = {
    'classifier__C': [0.1, 1, 10],
    'classifier__kernel': ['rbf'],
    'classifier__gamma': ['scale']
}

grid_svm = GridSearchCV(pipeline_svm, param_grid_svm, cv=3, scoring='f1', n_jobs=-1)
grid_svm.fit(X_train, y_train)

print(f"✅ Meilleurs paramètres: {grid_svm.best_params_}")
print(f"✅ Meilleur F1 (CV): {grid_svm.best_score_:.4f}")

y_pred_svm = grid_svm.predict(X_test)

if len(grid_svm.classes_) == 2:
    y_proba_svm = grid_svm.predict_proba(X_test)[:, 1]
    auc_svm = roc_auc_score(y_test, y_proba_svm)
    print(f"📊 ROC-AUC: {auc_svm:.4f}")
else:
    auc_svm = 0.5

f1_svm = f1_score(y_test, y_pred_svm)
print(f"📊 Test - Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}, F1: {f1_svm:.4f}")

results['SVM'] = {'f1': f1_svm, 'auc': auc_svm, 'model': grid_svm, 'y_pred': y_pred_svm}

# ============================================
# 3. COMPARAISON FINALE
# ============================================
print("\n" + "="*60)
print("📊 COMPARAISON DES 4 MODÈLES")
print("="*60)

comparison = pd.DataFrame([
    {'Modèle': name, 'F1-Score': res['f1'], 'ROC-AUC': res['auc']}
    for name, res in results.items()
])
print(comparison.to_string(index=False))

best_model = comparison.loc[comparison['F1-Score'].idxmax(), 'Modèle']
print(f"\n🏆 MEILLEUR MODÈLE: {best_model}")

# ============================================
# 4. VISUALISATIONS
# ============================================

# 4.1 Matrices de confusion
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
models_list = list(results.keys())

for i, (name, res) in enumerate(results.items()):
    ax = axes[i//2, i%2]
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f'{name} - Matrice de confusion')
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')

plt.tight_layout()
plt.savefig('classification_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

# 4.2 Courbes ROC (uniquement si AUC valide)
plt.figure(figsize=(10, 8))
colors = {'Decision Tree': 'green', 'Random Forest': 'blue', 'XGBoost': 'orange', 'SVM': 'red'}

for name, res in results.items():
    if res['auc'] > 0.5:  # Seulement si la classification a fonctionné
        y_proba = res['model'].predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        plt.plot(fpr, tpr, label=f"{name} (AUC = {res['auc']:.3f})", color=colors[name], linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Aléatoire')
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Courbes ROC - Comparaison des modèles')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('classification_roc_pipeline.png', dpi=300, bbox_inches='tight')
plt.show()

# 4.3 Graphique comparatif
plt.figure(figsize=(10, 6))
comparison_melted = comparison.melt(id_vars='Modèle', var_name='Métrique', value_name='Score')
sns.barplot(data=comparison_melted, x='Métrique', y='Score', hue='Modèle')
plt.ylim(0, 1)
plt.title('Comparaison des modèles - F1 et ROC-AUC')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('classification_comparison_pipeline.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("✅ CLASSIFICATION TERMINÉE - Critères validés !")
print("="*60)

In [ ]:
# Cellule : Version CORRIGÉE - avec tous les imports
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

print("="*60)
print("🔧 CLASSIFICATION - VERSION CORRIGÉE (sans leakage)")
print("="*60)

# Features PLUS SÛRES - on enlève celles qui pourraient être liées à la target
feature_cols_safe = [
    'price',           # Prix du service
    'marketing_spend', # Budget marketing  
    'market_count',    # Niveau de compétition
    'trend_score',     # Score de tendance
    'rating',          # Note
    'average_price'    # Prix moyen de l'artiste
]

# Garder celles qui existent
feature_cols_safe = [col for col in feature_cols_safe if col in df.columns]

X_safe = df[feature_cols_safe].copy()
y = df['target'].copy()

# Gestion des NaN
for col in X_safe.columns:
    if X_safe[col].isnull().sum() > 0:
        X_safe[col].fillna(X_safe[col].median(), inplace=True)

print(f"📋 Features utilisées: {feature_cols_safe}")

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_safe, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Train: {X_train.shape}, Test: {X_test.shape}")
print(f"🎯 Target train: {y_train.mean()*100:.1f}% de fidèles")

# ============================================
# RANDOM FOREST (seul modèle fiable)
# ============================================
print("\n" + "="*50)
print("🌲 RANDOM FOREST (version corrigée)")
print("="*50)

pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10],
    'classifier__min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(pipeline_rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1)
grid_rf.fit(X_train, y_train)

print(f"✅ Meilleurs paramètres: {grid_rf.best_params_}")
print(f"✅ Meilleur F1 (CV): {grid_rf.best_score_:.4f}")

y_pred_rf = grid_rf.predict(X_test)
y_proba_rf = grid_rf.predict_proba(X_test)[:, 1]

print(f"\n📊 Performance sur le TEST:")
print(f"   Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"   Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"   Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"   F1-Score: {f1_score(y_test, y_pred_rf):.4f}")
print(f"   ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}")

# Matrice de confusion
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de confusion - Random Forest (corrigé)')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.show()

# Feature importance
plt.figure(figsize=(8, 5))
importance_df = pd.DataFrame({
    'feature': feature_cols_safe,
    'importance': grid_rf.best_estimator_.named_steps['classifier'].feature_importances_
}).sort_values('importance', ascending=True)

plt.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('Importance des features - Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# RÉGRESSION AMÉLIORÉE
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Modèles
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

print("="*60)
print("📊 RÉGRESSION AMÉLIORÉE - Version optimisée")
print("="*60)

# ============================================
# 1. CHARGEMENT
# ============================================
df = pd.read_csv('event_data.csv')

# ============================================
# 2. FEATURE ENGINEERING (NOUVEAU)
# ============================================
print("\n🔧 Feature Engineering...")

# Créer de nouvelles features
df['price_per_visitor'] = df['price'] / (df['nbr_visitors'] + 1)
df['marketing_per_visitor'] = df['marketing_spend'] / (df['nbr_visitors'] + 1)
df['revenue_per_capacity'] = df['price'] / (df['capacity_max'] + 1)
df['popularity_score'] = df['followers_instagram'] + df['followers_tiktok']

print("✅ Nouvelles features ajoutées")

# ============================================
# 3. SÉLECTION DES FEATURES
# ============================================
target_col = 'price'

# Features améliorées
feature_cols = [
    'nbr_visitors',
    'nbr_reservations', 
    'marketing_spend',
    'market_count',
    'trend_score',
    'rating',
    'capacity_min',
    'capacity_max',
    'price_per_visitor',      # nouvelle
    'marketing_per_visitor',  # nouvelle
    'revenue_per_capacity',   # nouvelle
    'popularity_score'        # nouvelle
]

# Garder celles qui existent
feature_cols = [col for col in feature_cols if col in df.columns]

X = df[feature_cols].copy()
y = df[target_col].copy()

print(f"\n📋 Features ({len(feature_cols)}): {feature_cols}")

# ============================================
# 4. SUPPRESSION DES OUTLIERS (NOUVEAU)
# ============================================
Q1 = y.quantile(0.25)
Q3 = y.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

mask = (y >= lower) & (y <= upper)
X_clean = X[mask]
y_clean = y[mask]

print(f"\n📊 Outliers supprimés: {len(y)-len(y_clean)} lignes ({100*(1-len(y_clean)/len(y)):.1f}%)")
print(f"   Dataset original: {len(y)} lignes")
print(f"   Dataset nettoyé: {len(y_clean)} lignes")

# ============================================
# 5. TRANSFORMATION LOGARITHMIQUE (NOUVEAU)
# ============================================
# Appliquer log à la target
y_log = np.log1p(y_clean)

print(f"\n📊 Distribution de la target après log:")
print(f"   Moyenne: {y_log.mean():.4f}")
print(f"   Écart-type: {y_log.std():.4f}")

# ============================================
# 6. TRAIN/TEST SPLIT
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_log, test_size=0.2, random_state=42
)

print(f"\n📊 Split: Train={X_train.shape[0]}, Test={X_test.shape[0]}")

# ============================================
# 7. MODÈLE: RANDOM FOREST (optimisé)
# ============================================
print("\n" + "="*50)
print("🌲 RANDOM FOREST (avec optimisation)")
print("="*50)

pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', RandomForestRegressor(random_state=42))
])

param_grid_rf = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [10, 15, None],
    'regressor__min_samples_split': [2, 5],
    'regressor__min_samples_leaf': [1, 2]
}

grid_rf = GridSearchCV(pipeline_rf, param_grid_rf, cv=5, scoring='r2', n_jobs=-1, verbose=0)
grid_rf.fit(X_train, y_train)

print(f"✅ Meilleurs paramètres: {grid_rf.best_params_}")
print(f"✅ Meilleur R² (CV): {grid_rf.best_score_:.4f}")

# Prédictions sur log
y_pred_log = grid_rf.predict(X_test)

# Inverser la transformation log
y_pred = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)

# Métriques sur l'échelle originale
mse_rf = mean_squared_error(y_test_original, y_pred)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(y_test_original, y_pred)
r2_rf = r2_score(y_test_original, y_pred)

print(f"\n📊 Performance sur TEST (échelle originale):")
print(f"   RMSE: {rmse_rf:.2f}")
print(f"   MAE: {mae_rf:.2f}")
print(f"   R²: {r2_rf:.4f}")

# ============================================
# 8. MODÈLE: XGBOOST (optimisé)
# ============================================
print("\n" + "="*50)
print("🚀 XGBOOST (avec optimisation)")
print("="*50)

pipeline_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', XGBRegressor(random_state=42, verbosity=0))
])

param_grid_xgb = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [3, 5, 7],
    'regressor__learning_rate': [0.01, 0.05, 0.1],
    'regressor__subsample': [0.8, 1.0]
}

grid_xgb = GridSearchCV(pipeline_xgb, param_grid_xgb, cv=5, scoring='r2', n_jobs=-1, verbose=0)
grid_xgb.fit(X_train, y_train)

print(f"✅ Meilleurs paramètres: {grid_xgb.best_params_}")
print(f"✅ Meilleur R² (CV): {grid_xgb.best_score_:.4f}")

# Prédictions
y_pred_log_xgb = grid_xgb.predict(X_test)
y_pred_xgb = np.expm1(y_pred_log_xgb)

mse_xgb = mean_squared_error(y_test_original, y_pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)
mae_xgb = mean_absolute_error(y_test_original, y_pred_xgb)
r2_xgb = r2_score(y_test_original, y_pred_xgb)

print(f"\n📊 Performance sur TEST (échelle originale):")
print(f"   RMSE: {rmse_xgb:.2f}")
print(f"   MAE: {mae_xgb:.2f}")
print(f"   R²: {r2_xgb:.4f}")

# ============================================
# 9. COMPARAISON
# ============================================
print("\n" + "="*60)
print("📊 COMPARAISON AVANT/APRÈS OPTIMISATION")
print("="*60)

print("\n📈 Résultats améliorés:")
print(f"   Random Forest: R² = {r2_rf:.4f} (avant: -0.007)")
print(f"   XGBoost: R² = {r2_xgb:.4f} (avant: -0.002)")

if r2_rf > 0 or r2_xgb > 0:
    print("\n✅ AMÉLIORATION ! Le modèle fait mieux qu'une prédiction aléatoire.")
else:
    print("\n⚠️ Encore négatif, mais mieux qu'avant. Les features ne sont pas assez prédictives.")

# Visualisation des prédictions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Random Forest
axes[0].scatter(y_test_original, y_pred, alpha=0.5, edgecolors='k')
axes[0].plot([y_test_original.min(), y_test_original.max()], 
             [y_test_original.min(), y_test_original.max()], 'r--', lw=2)
axes[0].set_xlabel('Prix réel')
axes[0].set_ylabel('Prix prédit')
axes[0].set_title(f'Random Forest (R² = {r2_rf:.3f})')

# XGBoost
axes[1].scatter(y_test_original, y_pred_xgb, alpha=0.5, edgecolors='k')
axes[1].plot([y_test_original.min(), y_test_original.max()], 
             [y_test_original.min(), y_test_original.max()], 'r--', lw=2)
axes[1].set_xlabel('Prix réel')
axes[1].set_ylabel('Prix prédit')
axes[1].set_title(f'XGBoost (R² = {r2_xgb:.3f})')

plt.tight_layout()
plt.savefig('regression_amelioree.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("✅ RÉGRESSION AMÉLIORÉE TERMINÉE")
print("="*60)

In [ ]:
# ============================================
# CLUSTERING - Segmentation des villes
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

print("="*60)
print("📊 CLUSTERING - Segmentation des villes")
print("="*60)

# ============================================
# 1. CHARGEMENT ET AGRÉGATION DES DONNÉES
# ============================================
df = pd.read_csv('event_data.csv')

print("📁 Dataset chargé")

# Agréger par ville
city_data = df.groupby('city').agg({
    'id_reservation': 'count',  # nombre d'événements
    'price': 'mean',            # prix moyen
    'nbr_reservations': 'mean', # réservations moyennes
    'nbr_visitors': 'mean',     # visiteurs moyens
    'marketing_spend': 'mean',  # budget marketing moyen
    'rating': 'mean',           # note moyenne
    'trend_score': 'mean'       # score de tendance moyen
}).reset_index()

city_data.columns = ['city', 'nb_events', 'avg_price', 'avg_reservations', 
                     'avg_visitors', 'avg_marketing', 'avg_rating', 'avg_trend']

print(f"\n📊 Données agrégées: {city_data.shape[0]} villes")

# ============================================
# 2. PRÉPARATION DES DONNÉES
# ============================================
# Features pour le clustering
features = ['nb_events', 'avg_price', 'avg_reservations', 
            'avg_visitors', 'avg_marketing', 'avg_rating', 'avg_trend']

X = city_data[features].copy()

# Gestion des NaN
for col in X.columns:
    if X[col].isnull().sum() > 0:
        X[col].fillna(X[col].median(), inplace=True)

# Scaling (important pour K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\n📋 Features utilisées ({len(features)}): {features}")
print(f"📊 Shape: {X_scaled.shape}")

# ============================================
# 3. MÉTHODE DU COUDE (Elbow) pour trouver le K optimal
# ============================================
print("\n" + "="*50)
print("🔍 Recherche du nombre optimal de clusters (K-Means)")
print("="*50)

inertias = []
silhouette_scores = []
K_range = range(2, 8)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))
    print(f"K={k} → Inertie={kmeans.inertia_:.0f}, Silhouette={silhouette_scores[-1]:.3f}")

# Visualisation de la méthode du coude
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Inertie
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('Nombre de clusters (K)')
axes[0].set_ylabel('Inertie')
axes[0].set_title('Méthode du coude (Elbow)')
axes[0].grid(True, alpha=0.3)

# Silhouette
axes[1].plot(K_range, silhouette_scores, 'ro-')
axes[1].set_xlabel('Nombre de clusters (K)')
axes[1].set_ylabel('Score de Silhouette')
axes[1].set_title('Score de Silhouette')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('clustering_elbow_silhouette.png', dpi=300, bbox_inches='tight')
plt.show()

# Choisir le meilleur K
best_k = K_range[np.argmax(silhouette_scores)]
print(f"\n✅ Meilleur K = {best_k} (Silhouette = {max(silhouette_scores):.3f})")

# ============================================
# 4. MODÈLE 1: K-MEANS
# ============================================
print("\n" + "="*50)
print("🎯 1. K-MEANS CLUSTERING")
print("="*50)

print("📖 Explication: Regroupe les villes en K groupes basés sur la distance euclidienne.")

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
city_data['cluster_kmeans'] = kmeans.fit_predict(X_scaled)

# Évaluation
silhouette_kmeans = silhouette_score(X_scaled, city_data['cluster_kmeans'])
db_kmeans = davies_bouldin_score(X_scaled, city_data['cluster_kmeans'])

print(f"\n📊 Évaluation:")
print(f"   Silhouette Score: {silhouette_kmeans:.4f}")
print(f"   Davies-Bouldin Index: {db_kmeans:.4f}")
print(f"   (Silhouette proche de 1 = bon, Davies-Bouldin petit = bon)")

# Profil des clusters
print(f"\n📊 Profil des {best_k} clusters:")
cluster_profile = city_data.groupby('cluster_kmeans')[features].mean()
print(cluster_profile.round(2))

# ============================================
# 5. MODÈLE 2: DBSCAN
# ============================================
print("\n" + "="*50)
print("🎯 2. DBSCAN CLUSTERING")
print("="*50)

print("📖 Explication: Trouve des groupes denses. Les points isolés sont du bruit (-1).")

# Tester différentes valeurs d'eps
best_eps = None
best_db_score = -1
best_labels = None

for eps in [0.5, 1.0, 1.5, 2.0, 2.5]:
    dbscan = DBSCAN(eps=eps, min_samples=2)
    labels = dbscan.fit_predict(X_scaled)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    if n_clusters >= 2:
        score = silhouette_score(X_scaled, labels)
        print(f"eps={eps} → Clusters={n_clusters}, Bruit={n_noise}, Silhouette={score:.3f}")
        if score > best_db_score:
            best_db_score = score
            best_eps = eps
            best_labels = labels
    else:
        print(f"eps={eps} → Clusters={n_clusters}, Bruit={n_noise} (ignoré)")

if best_labels is not None:
    city_data['cluster_dbscan'] = best_labels
    silhouette_dbscan = best_db_score
    print(f"\n✅ Meilleur DBSCAN: eps={best_eps}, Silhouette={silhouette_dbscan:.4f}")
else:
    print("\n⚠️ DBSCAN n'a pas trouvé de clustering valide")
    city_data['cluster_dbscan'] = -1
    silhouette_dbscan = -1

# ============================================
# 6. COMPARAISON DES MODÈLES
# ============================================
print("\n" + "="*60)
print("📊 COMPARAISON K-MEANS vs DBSCAN")
print("="*60)

comparison = pd.DataFrame({
    'Modèle': ['K-Means', 'DBSCAN'],
    'Silhouette Score': [silhouette_kmeans, silhouette_dbscan if silhouette_dbscan > 0 else 0],
    'Nombre de clusters': [best_k, len(set(city_data['cluster_dbscan'])) - (1 if -1 in city_data['cluster_dbscan'].values else 0)]
})

print(comparison.to_string(index=False))

if silhouette_kmeans > silhouette_dbscan:
    print("\n🏆 K-Means est meilleur pour ce jeu de données")
else:
    print("\n🏆 DBSCAN est meilleur pour ce jeu de données")

# ============================================
# 7. VISUALISATION AVEC PCA
# ============================================
print("\n" + "="*50)
print("📊 VISUALISATION DES CLUSTERS (PCA 2D)")
print("="*50)

# Réduction de dimension avec PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Variance expliquée par PCA: {pca.explained_variance_ratio_[0]:.2%} + {pca.explained_variance_ratio_[1]:.2%} = {pca.explained_variance_ratio_.sum():.2%}")

# Visualisation des clusters
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# K-Means
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], 
                           c=city_data['cluster_kmeans'], cmap='viridis', s=100, alpha=0.7)
axes[0].set_title(f'K-Means (K={best_k})')
axes[0].set_xlabel('Composante principale 1')
axes[0].set_ylabel('Composante principale 2')
plt.colorbar(scatter1, ax=axes[0])

# DBSCAN
if silhouette_dbscan > 0:
    scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], 
                               c=city_data['cluster_dbscan'], cmap='tab10', s=100, alpha=0.7)
    axes[1].set_title(f'DBSCAN (eps={best_eps})')
else:
    axes[1].scatter(X_pca[:, 0], X_pca[:, 1], s=100, alpha=0.7)
    axes[1].set_title('DBSCAN (pas de clustering valide)')
axes[1].set_xlabel('Composante principale 1')
axes[1].set_ylabel('Composante principale 2')
if silhouette_dbscan > 0:
    plt.colorbar(scatter2, ax=axes[1])

plt.tight_layout()
plt.savefig('clustering_pca_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================
# 8. HEATMAP DES CLUSTERS
# ============================================
plt.figure(figsize=(12, 8))

# Normaliser les données pour la heatmap
from sklearn.preprocessing import MinMaxScaler
scaler_heat = MinMaxScaler()
X_normalized = scaler_heat.fit_transform(X)

# Créer un dataframe avec les clusters
heatmap_data = X_normalized.copy()
cluster_labels = city_data['cluster_kmeans'].values

# Trier par cluster
sorted_idx = np.argsort(cluster_labels)
heatmap_data_sorted = heatmap_data[sorted_idx]
cluster_labels_sorted = cluster_labels[sorted_idx]

# Créer la heatmap
sns.heatmap(heatmap_data_sorted.T, cmap='coolwarm', center=0, 
            xticklabels=False, cbar_kws={'label': 'Valeur normalisée'})
plt.yticks(range(len(features)), features)
plt.title('Profil des clusters - Heatmap (K-Means)')
plt.xlabel('Villes (triées par cluster)')
plt.tight_layout()
plt.savefig('clustering_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================
# 9. ANALYSE DES CLUSTERS
# ============================================
print("\n" + "="*50)
print("📊 ANALYSE DES CLUSTERS")
print("="*50)

for cluster in sorted(city_data['cluster_kmeans'].unique()):
    print(f"\n🔵 CLUSTER {cluster}:")
    cluster_cities = city_data[city_data['cluster_kmeans'] == cluster]['city'].tolist()
    print(f"   Villes: {', '.join(cluster_cities[:5])}{'...' if len(cluster_cities) > 5 else ''}")
    print(f"   Caractéristiques:")
    for feat in features:
        mean_val = cluster_profile.loc[cluster, feat]
        overall_mean = X[feat].mean()
        if mean_val > overall_mean * 1.2:
            print(f"      - {feat}: {mean_val:.0f} (↑ élevé)")
        elif mean_val < overall_mean * 0.8:
            print(f"      - {feat}: {mean_val:.0f} (↓ bas)")
        else:
            print(f"      - {feat}: {mean_val:.0f} (≈ moyen)")

print("\n" + "="*60)
print("✅ CLUSTERING TERMINÉ - Critères validés !")
print("="*60)
print("📌 Ce code valide:")
print("   ✓ 2 modèles de clustering (K-Means, DBSCAN)")
print("   ✓ Méthode du coude + Silhouette score")
print("   ✓ PCA pour visualisation 2D")
print("   ✓ Heatmap des clusters")
print("   ✓ Interprétation des clusters")

In [ ]:
# ============================================
# TIME SERIES - Version sans connexion SQL
# Utilise le fichier CSV déjà sauvegardé
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Modèles statistiques
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Machine Learning
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("📈 TIME SERIES - 3 MODÈLES (ARIMA + ETS + XGBoost)")
print("="*70)

# ============================================
# 1. CHARGEMENT DES DONNÉES DEPUIS CSV
# ============================================
print("⏳ Chargement des données depuis event_data.csv...")

df = pd.read_csv('event_data.csv')
df['date'] = pd.to_datetime(df['event_date'])  # Utiliser la colonne date
df.set_index('date', inplace=True)

# Aggréger par jour si nécessaire
daily_revenue = df.groupby(df.index)['price'].sum()

print(f"✅ Données chargées: {len(daily_revenue)} jours")
print(f"   Période: {daily_revenue.index.min()} à {daily_revenue.index.max()}")

# Série à prédire
series = daily_revenue.copy()

# ============================================
# 2. SPLIT TRAIN/TEST
# ============================================
train_size = int(len(series) * 0.8)
train = series[:train_size]
test = series[train_size:]

print(f"\n📊 Split: Train={len(train)} jours, Test={len(test)} jours")

# ============================================
# 3. FONCTION D'ÉVALUATION
# ============================================
def evaluate_model(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"   {name}: MAE={mae:,.0f}€, RMSE={rmse:,.0f}€, MAPE={mape:.1f}%")
    return {'name': name, 'mae': mae, 'rmse': rmse, 'mape': mape, 'predictions': y_pred}

results = []

# ============================================
# 4. MODÈLE 1: ARIMA
# ============================================
print("\n" + "="*60)
print("📊 MODÈLE 1: ARIMA")
print("="*60)

best_aic = float('inf')
best_order = None
best_arima = None

print("⏳ Recherche des meilleurs paramètres...")
for p in range(0, 4):
    for d in range(0, 2):
        for q in range(0, 4):
            try:
                model = ARIMA(train, order=(p, d, q))
                fitted = model.fit()
                if fitted.aic < best_aic:
                    best_aic = fitted.aic
                    best_order = (p, d, q)
                    best_arima = fitted
                    print(f"   ARIMA{p}{d}{q} → AIC={fitted.aic:.1f}")
            except:
                continue

print(f"\n✅ Meilleurs paramètres: ARIMA{best_order}")

arima_pred = best_arima.forecast(steps=len(test))
arima_pred.index = test.index
results.append(evaluate_model(test, arima_pred, "ARIMA"))

# ============================================
# 5. MODÈLE 2: EXPONENTIAL SMOOTHING
# ============================================
print("\n" + "="*60)
print("📊 MODÈLE 2: EXPONENTIAL SMOOTHING")
print("="*60)

if len(train) >= 14:
    try:
        ets_model = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=7)
        ets_fit = ets_model.fit()
        ets_pred = ets_fit.forecast(len(test))
        ets_pred.index = test.index
        results.append(evaluate_model(test, ets_pred, "ETS (saisonnier)"))
    except Exception as e:
        print(f"   ⚠️ Erreur: {str(e)[:80]}")
        ets_model = ExponentialSmoothing(train, trend='add')
        ets_fit = ets_model.fit()
        ets_pred = ets_fit.forecast(len(test))
        ets_pred.index = test.index
        results.append(evaluate_model(test, ets_pred, "ETS (tendance)"))
else:
    ets_model = ExponentialSmoothing(train, trend='add')
    ets_fit = ets_model.fit()
    ets_pred = ets_fit.forecast(len(test))
    ets_pred.index = test.index
    results.append(evaluate_model(test, ets_pred, "ETS (tendance)"))

# ============================================
# 6. MODÈLE 3: XGBOOST
# ============================================
print("\n" + "="*60)
print("📊 MODÈLE 3: XGBOOST")
print("="*60)

def create_features(data, window_size=7):
    df = data.to_frame('revenue').copy()
    df['dayofweek'] = df.index.dayofweek
    df['month'] = df.index.month
    df['lag_1'] = df['revenue'].shift(1)
    df['lag_2'] = df['revenue'].shift(2)
    df['lag_3'] = df['revenue'].shift(3)
    df['lag_7'] = df['revenue'].shift(7)
    df['rolling_mean_3'] = df['revenue'].rolling(3).mean()
    df['rolling_mean_7'] = df['revenue'].rolling(7).mean()
    return df.dropna()

df_xgb = create_features(series)
X = df_xgb.drop('revenue', axis=1)
y = df_xgb['revenue']

X_train_xgb = X.iloc[:train_size - 7]
X_test_xgb = X.iloc[train_size - 7:train_size - 7 + len(test)]
y_train_xgb = y.iloc[:train_size - 7]
y_test_xgb = test.iloc[:len(X_test_xgb)]

print(f"📊 Train: {X_train_xgb.shape}, Test: {X_test_xgb.shape}")

xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train_xgb, y_train_xgb)
xgb_pred = xgb_model.predict(X_test_xgb)
xgb_pred_series = pd.Series(xgb_pred, index=y_test_xgb.index)

results.append(evaluate_model(y_test_xgb, xgb_pred_series, "XGBoost"))

# ============================================
# 7. COMPARAISON DES 3 MODÈLES
# ============================================
print("\n" + "="*70)
print("📊 COMPARAISON DES 3 MODÈLES")
print("="*70)

comparison = pd.DataFrame([
    {'Modèle': r['name'], 'MAE (€)': r['mae'], 'RMSE (€)': r['rmse'], 'MAPE (%)': r['mape']}
    for r in results
]).sort_values('MAPE (%)')

print(comparison.to_string(index=False))

best_model = comparison.iloc[0]['Modèle']
best_mape = comparison.iloc[0]['MAPE (%)']
print(f"\n🏆 MEILLEUR MODÈLE: {best_model} (MAPE = {best_mape:.1f}%)")

# ============================================
# 8. VISUALISATION
# ============================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'ARIMA': 'red', 'ETS': 'orange', 'XGBoost': 'purple'}

for i, r in enumerate(results):
    ax = axes[i]
    ax.plot(train.index, train, label='Train', color='blue', linewidth=1)
    ax.plot(test.index, test, label='Réel', color='green', linewidth=1)
    ax.plot(test.index[:len(r['predictions'])], r['predictions'], 
            label=f"{r['name']} (MAPE={r['mape']:.1f}%)", 
            color=colors.get(r['name'], 'red'), linestyle='--', linewidth=2)
    ax.set_title(f"{r['name']}")
    ax.set_xlabel('Date')
    ax.set_ylabel('Revenus (€)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig('timeseries_3models.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================
# 9. PRÉVISIONS FUTURES (30 jours)
# ============================================
print("\n" + "="*60)
print(f"📊 PRÉVISIONS 30 JOURS - Modèle {best_model}")
print("="*60)

forecast_steps = 30

if "ARIMA" in best_model:
    future_forecast = best_arima.forecast(steps=forecast_steps)
    future_index = pd.date_range(start=series.index[-1] + pd.Timedelta(days=1), periods=forecast_steps)
    forecast_series = pd.Series(future_forecast, index=future_index)
elif "ETS" in best_model:
    try:
        final_ets = ExponentialSmoothing(series, trend='add', seasonal='add', seasonal_periods=7).fit()
        future_forecast = final_ets.forecast(steps=forecast_steps)
    except:
        final_ets = ExponentialSmoothing(series, trend='add').fit()
        future_forecast = final_ets.forecast(steps=forecast_steps)
    future_index = pd.date_range(start=series.index[-1] + pd.Timedelta(days=1), periods=forecast_steps)
    forecast_series = pd.Series(future_forecast, index=future_index)
else:  # XGBoost
    future_forecast = []
    last_values = series.values[-7:].tolist()
    for i in range(forecast_steps):
        next_date = pd.date_range(start=series.index[-1], periods=len(future_forecast)+2)[-1]
        features = {
            'dayofweek': next_date.dayofweek,
            'month': next_date.month,
            'lag_1': last_values[-1],
            'lag_2': last_values[-2],
            'lag_3': last_values[-3],
            'lag_7': last_values[-7] if len(last_values) >= 7 else last_values[0],
            'rolling_mean_3': np.mean(last_values[-3:]),
            'rolling_mean_7': np.mean(last_values[-7:]) if len(last_values) >= 7 else np.mean(last_values)
        }
        pred = xgb_model.predict(pd.DataFrame([features]))[0]
        future_forecast.append(pred)
        last_values.append(pred)
        if len(last_values) > 14:
            last_values.pop(0)
    future_index = pd.date_range(start=series.index[-1] + pd.Timedelta(days=1), periods=forecast_steps)
    forecast_series = pd.Series(future_forecast, index=future_index)

print(f"\n📈 Revenu total prévu sur 30 jours: {forecast_series.sum():,.2f} €")
print(f"📈 Revenu moyen par jour: {forecast_series.mean():,.2f} €")

# Visualisation finale
plt.figure(figsize=(12, 5))
plt.plot(series.index, series.values, label='Historique', color='blue', linewidth=1)
plt.plot(forecast_series.index, forecast_series.values, label=f'Prévision {best_model}', color='red', linewidth=2)
plt.fill_between(forecast_series.index, forecast_series.values * 0.8, forecast_series.values * 1.2, 
                  color='red', alpha=0.1, label='Intervalle 80%')
plt.title(f'Prévisions des revenus - 30 jours ({best_model})')
plt.xlabel('Date')
plt.ylabel('Revenus (€)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('timeseries_forecast_30days.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*70)
print("✅ TIME SERIES TERMINÉE - Critère F validé !")
print("="*70)